# Efficient Drug Discovery using Molecular Data
## Notebook 3: Deep Neural Network Training & Evaluation
**Author:** Dev Kapania | IIT Roorkee Research Intern

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print('Libraries loaded!')

## 1. Load Processed Data

In [ ]:
X_train = np.load('../data/processed/X_train.npy')
X_val   = np.load('../data/processed/X_val.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')
y_test  = np.load('../data/processed/y_test.npy')

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
input_dim = X_train.shape[1]
print(f'Input dimension: {input_dim}')

## 2. Build Deep Neural Network

In [ ]:
def build_dnn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(1, activation='sigmoid')
    ], name='DrugActivityDNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model = build_dnn(input_dim)
model.summary()

## 3. Train with Callbacks

In [ ]:
cb = [
    callbacks.EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6),
    callbacks.ModelCheckpoint('../models/best_model.h5', monitor='val_auc',
                              save_best_only=True, mode='max')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=cb,
    verbose=1
)

print(f'\nTraining complete! Best val AUC: {max(history.history["val_auc"]):.4f}')

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, metric, title in zip(axes,
    ['loss', 'accuracy', 'auc'],
    ['Loss', 'Accuracy', 'ROC-AUC']):
    ax.plot(history.history[metric],       label='Train', color='#8e44ad')
    ax.plot(history.history[f'val_{metric}'], label='Val',   color='#2980b9', linestyle='--')
    ax.set_title(f'Training vs Validation {title}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Final Evaluation on Test Set

In [ ]:
y_prob = model.predict(X_test).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Inactive', 'Active']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Inactive', 'Active'],
            yticklabels=['Inactive', 'Active'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#8e44ad', lw=2,
             label=f'ROC AUC = {roc_auc_score(y_test, y_prob):.2f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Final Model

In [ ]:
model.save('../models/drug_dnn_final.h5')
print('Model saved to ../models/drug_dnn_final.h5')
print('Done!')